In [1]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [2]:
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
import tiktoken

c:\Users\RakeshVasanthKimidi\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
Data = load_dataset("csv",data_files={"train":"dataset\\train.csv","validation":"dataset\\validation.csv"})

In [4]:
print(Data)

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2119719
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 21990
    })
})


In [5]:
Data["train"][:5]

{'text': ['One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.',
  'Once upon a time, there was a little car named Beep. Beep loved to go fast and play in the sun. Beep was a healthy car because he always had good fuel. Good fuel made Beep happy and strong.\n\nOne day, Beep was driving in the park when he saw a big tree. The tree had many leav

In [6]:
encoder = tiktoken.get_encoding("gpt2")

In [7]:
vocab_size = encoder.n_vocab

In [8]:
encoder.eot_token

50256

In [9]:
def encoding(example):
    ids = encoder.encode_ordinary(example["text"])
    out = {"ids":ids,"len":len(ids)}
    return out

In [10]:
missing = sum(1 for row in Data["train"] if row ["text"] == None or row["text"].strip() == "")
missing

230

In [11]:
for split in Data.keys():
    Data[split] = Data[split].filter(lambda x : x["text"] is not None and x["text"].strip() != "")

In [12]:
missing = sum(1 for row in Data["train"] if row ["text"] == None or row["text"].strip() == "")
missing

0

In [13]:
data = Data.map(encoding)

In [14]:
data["train"][:5]

{'text': ['One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.',
  'Once upon a time, there was a little car named Beep. Beep loved to go fast and play in the sun. Beep was a healthy car because he always had good fuel. Good fuel made Beep happy and strong.\n\nOne day, Beep was driving in the park when he saw a big tree. The tree had many leav

In [15]:
data = data.remove_columns("text")

In [16]:
data["train"][:5]

{'ids': [[3198,
   1110,
   11,
   257,
   1310,
   2576,
   3706,
   20037,
   1043,
   257,
   17598,
   287,
   607,
   2119,
   13,
   1375,
   2993,
   340,
   373,
   2408,
   284,
   711,
   351,
   340,
   780,
   340,
   373,
   7786,
   13,
   20037,
   2227,
   284,
   2648,
   262,
   17598,
   351,
   607,
   1995,
   11,
   523,
   673,
   714,
   34249,
   257,
   4936,
   319,
   607,
   10147,
   13,
   198,
   198,
   43,
   813,
   1816,
   284,
   607,
   1995,
   290,
   531,
   11,
   366,
   29252,
   11,
   314,
   1043,
   428,
   17598,
   13,
   1680,
   345,
   2648,
   340,
   351,
   502,
   290,
   34249,
   616,
   10147,
   1701,
   2332,
   1995,
   13541,
   290,
   531,
   11,
   366,
   5297,
   11,
   20037,
   11,
   356,
   460,
   2648,
   262,
   17598,
   290,
   4259,
   534,
   10147,
   526,
   198,
   198,
   41631,
   11,
   484,
   4888,
   262,
   17598,
   290,
   384,
   19103,
   262,
   4936,
   319,
   20037,
   338,
   10147,
   1

In [17]:
for split,dset in data.items():
    ary_len = np.sum(dset["len"])
    print(ary_len)
    filename = f"{split}.bin"
    arry = np.memmap(filename=filename,dtype=np.uint16,mode="w+",shape=(ary_len,))
    total_batches = 1024
    indx = 0
    for batch_indx in range(total_batches):
        batch = dset.shard(num_shards=total_batches,index=batch_indx,contiguous=True).with_format("numpy")
        arr_batch = np.concatenate(batch["ids"])
        print(len(arr_batch))

        arry[indx:indx+len(arr_batch)] = arr_batch
        indx += len(arr_batch)
    arry.flush()
             

471872517
431906
424907
425744
496328
437836
477616
549013
389417
496140
469058
417134
473395
458463
439946
463461
510651
476215
422657
494976
445509
424472
457894
416299
461661
429050
446201
511908
443029
487242
491764
421787
457162
491336
430709
400628
470538
464057
456876
372881
463600
457544
439685
380814
415273
464664
432758
506803
425575
441997
438451
460009
511256
464501
579663
440163
452118
468116
456699
447587
492581
439917
481977
459129
502284
477027
517187
419724
449131
464426
487351
486353
494314
449299
429255
478831
521873
502330
500095
486708
465830
528356
578684
431866
479581
498692
419568
438287
493456
477551
411647
427301
484266
477675
542490
475445
450634
452941
465460
522570
422653
479440
481930
440953
490824
444773
447414
418991
443094
446377
487234
565720
443712
513979
465971
509951
462470
447724
462464
469099
576698
467720
434357
454830
423293
504597
509225
494697
435786
380343
431658
524762
547735
442953
475895
433046
404572
405508
455837
439279
534890
479433
470

In [18]:
block_size = 128
batch = 3
def get_split(split):
    if split == "train":
        Xdata = np.memmap("train.bin",mode="r",dtype=np.uint16)
    else:
        Xdata = np.memmap("validation.bin",mode="r",dtype=np.uint16)
    
    ix = torch.randint(len(Xdata)-block_size,(batch,))
    x = torch.stack([torch.from_numpy((Xdata[i:i+block_size]).astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy((Xdata[i+1:i+block_size+1]).astype(np.int64)) for i in ix])
    
    if device =="cuda":
        x,y = x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)
    else:
        x,y = x.to(device),y.to(device)
    
    return x,y


In [19]:
class Embedding(nn.Module):
  def __init__(self,emb_dim,vocab_size,block_size):
    super().__init__()

    self.emb_dim = emb_dim
    self.vocab_size = vocab_size
    self.block_size = block_size
    self.emb = nn.Embedding(self.vocab_size,self.emb_dim)
    self.pos = nn.Embedding(self.block_size,self.emb_dim)
  def forward(self,x):
    B,T = x.shape
    positions = torch.arange(T,device=x.device)

    token_embd = self.emb(x)
    pos_emb = self.pos(positions)

    return token_embd+pos_emb

class CausalAttention(nn.Module):
  def __init__(self,emb_dim,head_dim,block_size):
    super().__init__()
    self.emb_dim = emb_dim
    self.head_dim = head_dim
    self.block_size = block_size

    self.wq = nn.Linear(self.emb_dim,self.head_dim,bias=False)
    self.wk = nn.Linear(self.emb_dim,self.head_dim,bias=False)
    self.wv = nn.Linear(self.emb_dim,self.head_dim,bias=False)
    self.register_buffer("tril",torch.tril(torch.ones(self.block_size,self.block_size)))
    self.Dropout = nn.Dropout(0.2)
  def forward(self,x):
    B,T,C = x.shape
    Q = self.wq(x)
    K = self.wk(x)
    V = self.wv(x)

    scores = (Q @ K.transpose(-2,-1))/(self.head_dim**0.5)
    scores = scores.masked_fill(self.tril[:T,:T] == 0,float('-inf'))
    weights = F.softmax(scores,dim=-1)
    weights = self.Dropout(weights)
    out = weights @ V

    return out

class MultiHeadAttention(nn.Module):
  def __init__(self,emb_dim,num_heads,block_size):
    super().__init__()
    self.emb_dim = emb_dim
    self.num_heads = num_heads
    self.head_dim = emb_dim // num_heads
    self.block_size = block_size

    self.heads = nn.ModuleList(
        [
            CausalAttention(self.emb_dim,self.head_dim,self.block_size) for _ in range(self.num_heads)

        ]
    )
    self.proj = nn.Linear(self.emb_dim,self.emb_dim)
  def forward(self,x):
    out = torch.cat([h(x) for h in self.heads],dim=-1)
    out = self.proj(out)
    return out
class FeedForward(nn.Module):
  def __init__(self,emb_dim):
    super().__init__()
    self.emb_dim = emb_dim

    self.ffn = nn.Sequential(
        nn.Linear(self.emb_dim,4*self.emb_dim),
        nn.GELU(),
        nn.Linear(4*self.emb_dim,self.emb_dim)

    )
  def forward(self,x):
    out = self.ffn(x)
    return out
class TransformerBlock(nn.Module):
  def __init__(self,emb_dim,num_heads,block_size):
    super().__init__()
    self.emb_dim = emb_dim
    self.num_heads = num_heads
    self.block_size = block_size
    self.head_dim = emb_dim // num_heads

    self.LayerNorm1 = nn.LayerNorm(self.emb_dim)
    self.LayerNorm2 = nn.LayerNorm(self.emb_dim)
    self.att = MultiHeadAttention(self.emb_dim,self.num_heads,self.block_size)
    self.ffn = FeedForward(self.emb_dim)

  def forward(self,x):
    x = x + self.att(self.LayerNorm1(x))
    x = x + self.ffn(self.LayerNorm2(x))

    return x
class GPT(nn.Module):
  def __init__(self,vocab_size,emb_dim,num_heads,block_size,num_layers):
    super().__init__()

    self.vocab_size = vocab_size
    self.emb_dim = emb_dim
    self.num_heads = num_heads
    self.block_size = block_size
    self.num_layers = num_layers

    self.embedding_layer = Embedding(self.emb_dim,self.vocab_size,self.block_size)
    self.Transformer_blocks = nn.Sequential(
        *[
            TransformerBlock(self.emb_dim,self.num_heads,self.block_size) for _ in range(self.num_layers)
        ]
    )

    self.ln_f = nn.LayerNorm(self.emb_dim)
    self.lm_head = nn.Linear(self.emb_dim,self.vocab_size)
  def forward(self,x,targets=None):
    x = self.embedding_layer(x)
    x = self.Transformer_blocks(x)
    x = self.ln_f(x)
    logits  = self.lm_head(x)

    loss = None

    if targets is not None:
      B , T , C = logits.shape
      logits_flat = logits.view(B*T,C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits_flat,targets)

    return logits , loss

  def generate(self,idx,max_new_tokens):
    for _ in range(max_new_tokens):
      idx_cond = idx[:, -self.block_size:]
      logits , loss = self(idx_cond)
      logits = logits[:,-1,:]
      probs = F.softmax(logits,dim=-1)
      idx_next = torch.multinomial(probs,num_samples=1)
      idx = torch.cat((idx,idx_next),dim=1)
    return idx




In [20]:
model = GPT(
    vocab_size=vocab_size,
    emb_dim=384,
    block_size=block_size,
    num_heads=12,
    num_layers=12
).to(device)



In [21]:
print(sum(p.numel() for p in model.parameters())/1e6, 'M parameters')

59.977297 M parameters


In [22]:
xb, yb = get_split("train")

logits, loss = model(xb, yb)

print(logits.shape)
print(loss)

torch.Size([3, 128, 50257])
tensor(10.9924, device='cuda:0', grad_fn=<NllLossBackward0>)


In [23]:
optimizer = torch.optim.AdamW(model.parameters(),lr=1e-4)

In [24]:
max_iterations = 20000
eval_interval = 500

In [25]:
@torch.no_grad()
def loss_estimate():
  model.eval()
  out = {}
  for split in ["train","test"]:
    losses = torch.zeros(100)
    for k in range(100):
        xb,yb = get_split(split)
        _,loss = model(xb,yb)
        losses[k] = loss.item()
    out[split] = losses.mean()
  model.train()
  return out


In [26]:
for step in range(max_iterations):
  if step % eval_interval == 0 or step == max_iterations - 1:
    losses = loss_estimate()
    print(
            f"Step {step:5d} | "
            f"Train Loss: {losses['train']:.4f} | "
            f"Val Loss: {losses['test']:.4f}"
        )
  xb,yb = get_split("train")
  logits,loss = model(xb,yb)
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

Step     0 | Train Loss: 11.0275 | Val Loss: 11.0271
Step   500 | Train Loss: 4.6429 | Val Loss: 4.6236
Step  1000 | Train Loss: 4.2572 | Val Loss: 4.2164
Step  1500 | Train Loss: 3.9880 | Val Loss: 4.0482
Step  2000 | Train Loss: 3.8209 | Val Loss: 3.8439
Step  2500 | Train Loss: 3.6933 | Val Loss: 3.7199
Step  3000 | Train Loss: 3.5582 | Val Loss: 3.6169
Step  3500 | Train Loss: 3.5302 | Val Loss: 3.5016
Step  4000 | Train Loss: 3.3975 | Val Loss: 3.4283
Step  4500 | Train Loss: 3.3283 | Val Loss: 3.3457
Step  5000 | Train Loss: 3.2773 | Val Loss: 3.3006
Step  5500 | Train Loss: 3.2041 | Val Loss: 3.2003
Step  6000 | Train Loss: 3.1831 | Val Loss: 3.1780
Step  6500 | Train Loss: 3.1433 | Val Loss: 3.1178
Step  7000 | Train Loss: 3.1573 | Val Loss: 3.0861
Step  7500 | Train Loss: 3.0527 | Val Loss: 3.0380
Step  8000 | Train Loss: 3.0318 | Val Loss: 3.0543
Step  8500 | Train Loss: 2.9875 | Val Loss: 3.0328
Step  9000 | Train Loss: 2.9768 | Val Loss: 2.9013
Step  9500 | Train Loss: 2.98

In [27]:
text = "Once upon a time there was a pumpkin"
input_test = torch.tensor(encoder.encode_ordinary(text)).to(device).unsqueeze(0)
input_test

tensor([[ 7454,  2402,   257,   640,   612,   373,   257, 30089]],
       device='cuda:0')

In [28]:
torch.zeros((1,1),dtype=torch.long,device=device)

tensor([[0]], device='cuda:0')

In [29]:
gen = model.generate(input_test,max_new_tokens=2000).tolist()

In [30]:
encoder.decode(gen[0])

'Once upon a time there was a pumpkin. Every day the suitcase and the other children went over to the ocean. One day, a little bird came to visit Lily\'s room. The bird was scared and said, "Don\'t worry, so you\'ll just help me find my boat to fix." \n\nLily turned around and watched the surprises. She asked a little girl, "Ben, can I create a picture?" \n\nMom said, "I love you too, right! answering gate is peaceful and thinks of a very long day."\n\nAs the bird took a few days chains to help her with his friends. They all said, "We will find fresh pastry we\'re hungry. We need to make up." \n\nAfter they finished looking for a while, a climbing down to take off buckets. Lily\'s heart was scared and delicate. Timmy wrote all of apples for his dad. He asked a 3 year old, "I\'m proud of you for pushing our company forgirl." \n\n\nLily learned an idea. She thought about this idea, the story: Baren who can help all kinds of tight instead. She had an amazing job about all the really hard 